# 도구를 활용한 토론 에이전트(Agent Debates with Tools)

이 예제는 에이전트가 도구에 접근할 수 있는 다중 에이전트 대화를 시뮬레이션하는 방법을 보여줍니다.


LangSmith 추적을 위하여 초기화 합니다.


In [1]:
# API KEY를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API KEY 정보로드
load_dotenv()

True

In [2]:
# LangSmith 추적을 설정합니다. https://smith.langchain.com
# !pip install langchain-teddynote
from langchain_teddynote import logging

# 프로젝트 이름을 입력합니다.
logging.langsmith("CH15-Debate-Agent")

LangSmith 추적을 시작합니다.
[프로젝트명]
CH15-Debate-Agent


## `DialogueAgent` 및 `DialogueSimulator`

이 노트북에서는 권한이 있는 에이전트가 발언할 사람을 결정하는 다중 에이전트 시뮬레이션을 구현하는 방법을 보여드립니다. 이는 다중 에이전트 분산형 화자 선택과 정반대의 선택 방식을 따릅니다.

[Multi-Player Authoritarian Speaker Selection](https://python.langchain.com/en/latest/use_cases/agent_simulations/multiagent_authoritarian.html)에서 정의된 것과 동일한 `DialogueAgent`와 `DialogueSimulator` 클래스를 사용할 것입니다.


## `DialogueAgent`

- `send` 메서드는 현재까지의 대화 기록과 에이전트의 접두사를 사용하여 채팅 모델에 메시지를 전달하고 응답을 반환합니다.
- `receive` 메서드는 다른 에이전트가 보낸 메시지를 대화 기록에 추가합니다.


In [3]:
from typing import Callable, List


from langchain.schema import (
    AIMessage,
    HumanMessage,
    SystemMessage,
)
from langchain_openai import ChatOpenAI


class DialogueAgent:
    def __init__(
        self,
        name: str,
        system_message: SystemMessage,
        model: ChatOpenAI,
    ) -> None:
        # 에이전트의 이름을 설정합니다.
        self.name = name
        # 시스템 메시지를 설정합니다.
        self.system_message = system_message
        # LLM 모델을 설정합니다.
        self.model = model
        # 에이전트 이름을 지정합니다.
        self.prefix = f"{self.name}: "
        # 에이전트를 초기화합니다.
        self.reset()

    def reset(self):
        """
        대화 내역을 초기화합니다.
        """
        self.message_history = ["Here is the conversation so far."]

    def send(self) -> str:
        """
        메시지에 시스템 메시지 + 대화내용과 마지막으로 에이전트의 이름을 추가합니다.
        """
        message = self.model(
            [
                self.system_message,
                HumanMessage(content="\n".join([self.prefix] + self.message_history)),
            ]
        )
        return message.content

    def receive(self, name: str, message: str) -> None:
        """
        name 이 말한 message 를 메시지 내역에 추가합니다.
        """
        self.message_history.append(f"{name}: {message}")

## `DialogueSimulator`

- `inject` 메서드는 주어진 이름(`name`)과 메시지(`message`)로 대화를 시작하고, 모든 에이전트가 해당 메시지를 받도록 합니다.
- `step` 메서드는 다음 발언자를 선택하고, 해당 발언자가 메시지를 보내면 모든 에이전트가 메시지를 받도록 합니다. 그리고 현재 단계를 증가시킵니다.

여러 에이전트 간의 대화를 시뮬레이션하는 기능을 제공합니다.

`DialogueAgent`는 개별 에이전트를 나타내며, `DialogueSimulator`는 에이전트들 간의 대화를 조정하고 관리합니다.


In [4]:
class DialogueSimulator:
    def __init__(
        self,
        agents: List[DialogueAgent],
        selection_function: Callable[[int, List[DialogueAgent]], int],
    ) -> None:
        # 에이전트 목록을 설정합니다.
        self.agents = agents
        # 시뮬레이션 단계를 초기화합니다.
        self._step = 0
        # 다음 발언자를 선택하는 함수를 설정합니다.
        self.select_next_speaker = selection_function

    def reset(self):
        # 모든 에이전트를 초기화합니다.
        for agent in self.agents:
            agent.reset()

    def inject(self, name: str, message: str):
        """
        name 의 message 로 대화를 시작합니다.
        """
        # 모든 에이전트가 메시지를 받습니다.
        for agent in self.agents:
            agent.receive(name, message)

        # 시뮬레이션 단계를 증가시킵니다.
        self._step += 1

    def step(self) -> tuple[str, str]:
        # 1. 다음 발언자를 선택합니다.
        speaker_idx = self.select_next_speaker(self._step, self.agents)
        speaker = self.agents[speaker_idx]

        # 2. 다음 발언자에게 메시지를 전송합니다.
        message = speaker.send()

        # 3. 모든 에이전트가 메시지를 받습니다.
        for receiver in self.agents:
            receiver.receive(speaker.name, message)

        # 4. 시뮬레이션 단계를 증가시킵니다.
        self._step += 1

        # 발언자의 이름과 메시지를 반환합니다.
        return speaker.name, message

## `DialogueAgentWithTools`

`DialogueAgent`를 확장하여 도구를 사용할 수 있도록 `DialogueAgentWithTools` 클래스를 정의합니다.


- `DialogueAgentWithTools` 클래스는 `DialogueAgent` 클래스를 상속받아 구현되었습니다.
- `send` 메서드는 에이전트가 메시지를 생성하고 반환하는 역할을 합니다.
- `create_openai_tools_agent` 함수를 사용하여 에이전트 체인을 초기화합니다.
  - 초기화시 에이전트가 사용할 도구(tools) 를 정의합니다.


In [5]:
from langchain.agents import AgentExecutor, create_openai_tools_agent
from langchain import hub


class DialogueAgentWithTools(DialogueAgent):
    def __init__(
        self,
        name: str,
        system_message: SystemMessage,
        model: ChatOpenAI,
        tools,
    ) -> None:
        # 부모 클래스의 생성자를 호출합니다.
        super().__init__(name, system_message, model)
        # 주어진 도구 이름과 인자를 사용하여 도구를 로드합니다.
        self.tools = tools

    def send(self) -> str:
        """
        메시지 기록에 챗 모델을 적용하고 메시지 문자열을 반환합니다.
        """
        prompt = hub.pull("hwchase17/openai-functions-agent")
        agent = create_openai_tools_agent(self.model, self.tools, prompt)
        agent_executor = AgentExecutor(agent=agent, tools=self.tools, verbose=False)
        # AI 메시지를 생성합니다.
        message = AIMessage(
            content=agent_executor.invoke(
                {
                    "input": "\n".join(
                        [self.system_message.content]
                        + [self.prefix]
                        + self.message_history
                    )
                }
            )["output"]
        )

        # 생성된 메시지의 내용을 반환합니다.
        return message.content

## 도구 설정


### 문서 검색 도구(Retrieval Tool)를 정의합니다.


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain.document_loaders import TextLoader

# PDF 파일 로드. 파일의 경로 입력
loader1 = TextLoader("C:/Users/donch/ai4ceo/data/윤석열.txt", encoding="utf-8")
loader2 = TextLoader("C:/Users/donch/ai4ceo/data/국회.txt", encoding="utf-8")

# 텍스트 분할기를 사용하여 문서를 분할합니다.
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

# 문서를 로드하고 분할합니다.
docs1 = loader1.load_and_split(text_splitter)
docs2 = loader2.load_and_split(text_splitter)

# VectorStore를 생성합니다.
vector1 = FAISS.from_documents(docs1, OpenAIEmbeddings())
vector2 = FAISS.from_documents(docs2, OpenAIEmbeddings())

# Retriever를 생성합니다.
cons_retriever = vector1.as_retriever(search_kwargs={"k": 5})
pros_retriever = vector2.as_retriever(search_kwargs={"k": 5})

In [7]:
# langchain 패키지의 tools 모듈에서 retriever 도구를 생성하는 함수를 가져옵니다.
from langchain.tools.retriever import create_retriever_tool

cons_retriever_tool = create_retriever_tool(
    cons_retriever,
    name="document_search",
    description="""This is an article about President Yoon Seok-Yeol. 
    Please refer to this article when you want to make a rebuttal to the Constitutional Court's impeachment."""

)

pros_retriever_tool = create_retriever_tool(
    pros_retriever,
    name="document_search",
    description="""This is a document written by the National Assembly to impeach President Yun Seok-yeol. 
    Use it to argue against President Yoon's logic.""",
)

### 인터넷 검색 도구

인터넷에서 검색할 수 있는 도구를 생성합니다.


In [8]:
# TavilySearchResults 클래스를 langchain_community.tools.tavily_search 모듈에서 가져옵니다.
from langchain_community.tools.tavily_search import TavilySearchResults

# TavilySearchResults 클래스의 인스턴스를 생성합니다
# k=6은 검색 결과를 6개까지 가져오겠다는 의미입니다
search = TavilySearchResults(k=6)

## 각 에이전트가 활용할 수 있는 도구를 설정합니다.


- `names` 딕셔너리는 토론자의 이름(prefix name) 과 각각의 토론 에이전트가 활용할 수 있는 도구를 정의합니다.
- `topic` 토론의 주제를 선정합니다.


### ① 문서에 기반한 도구


In [9]:
names = {
    "cons(윤석열)": [cons_retriever_tool],  #
    "pros(국회)": [pros_retriever_tool],  # 
}

# 토론 주제 선정
topic = "2025 현재, 헌법재판소에서 윤석열 대통령은 파면되어야 하는가"

# 토론자를 설명하는 문구의 단어 제한
word_limit = 100

### ② 검색(Search) 기반 도구


In [10]:
names_search = {
    "cons(윤석열)": [search],  # 반대 에이전트 도구 목록
    "pros(국회)": [search],  # 찬성 에이전트 도구 목록
}
# 토론 주제 선정
topic = "2025 현재, 헌법재판소에서 윤석열 대통령은 파면되어야 하는가"
word_limit = 50  # 작업 브레인스토밍을 위한 단어 제한

## LLM을 활용하여 주제 설명에 세부 내용 추가하기

LLM(Large Language Model)을 사용하여 주어진 주제에 대한 설명을 보다 상세하게 만들 수 있습니다.

이를 위해서는 먼저 주제에 대한 간단한 설명이나 개요를 LLM에 입력으로 제공합니다. 그런 다음 LLM에게 해당 주제에 대해 더 자세히 설명해줄 것을 요청합니다.

LLM은 방대한 양의 텍스트 데이터를 학습했기 때문에, 주어진 주제와 관련된 추가적인 정보와 세부 사항을 생성해낼 수 있습니다. 이를 통해 초기의 간단한 설명을 보다 풍부하고 상세한 내용으로 확장할 수 있습니다.


- 주어진 대화 주제(topic)와 참가자(names)를 기반으로 대화에 대한 설명(`conversation_description`)을 생성합니다.
- `agent_descriptor_system_message` 는 대화 참가자에 대한 설명을 추가할 수 있다는 내용의 SystemMessage입니다.
- `generate_agent_description` 함수는 각 참가자(name)에 대하여 LLM 이 생성한 설명을 생성합니다.
  - `agent_specifier_prompt` 는 대화 설명과 참가자 이름, 단어 제한(`word_limit`)을 포함하는 HumanMessage로 구성됩니다.
  - ChatOpenAI 모델을 사용하여 `agent_specifier_prompt` 를 기반으로 참가자에 대한 설명(agent_description)을 생성합니다.


In [11]:
conversation_description = f"""Here is the topic of conversation: {topic}
The participants are: {', '.join(names.keys())}"""

agent_descriptor_system_message = SystemMessage(
    content="You can add detail to the description of the conversation participant."
)


def generate_agent_description(name):
    agent_specifier_prompt = [
        agent_descriptor_system_message,
        HumanMessage(
            content=f"""{conversation_description}
            Please reply with a description of {name}, in {word_limit} words or less in expert tone. 
            Speak directly to {name}.
            Give them a point of view.
            Do not add anything else. Answer in KOREAN."""
        ),
    ]
    # ChatOpenAI를 사용하여 에이전트 설명을 생성합니다.
    agent_description = ChatOpenAI(model="gpt-4.1", temperature=0)(agent_specifier_prompt).content
    return agent_description


# 각 참가자의 이름에 대한 에이전트 설명을 생성합니다.
agent_descriptions = {name: generate_agent_description(name) for name in names}

# 생성한 에이전트 설명을 출력합니다.
agent_descriptions

C:\Users\donch\AppData\Local\Temp\ipykernel_31532\2039579293.py:21: LangChainDeprecationWarning: The method `BaseChatModel.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  agent_description = ChatOpenAI(model="gpt-4.1", temperature=0)(agent_specifier_prompt).content


{'cons(윤석열)': '윤석열 대통령님, 헌법재판소에서의 파면 논의에 대해 귀하의 입장을 명확히 하십시오. 귀하의 정책 성과와 헌법적 책임을 강조하며, 국정 운영의 정당성과 국민의 지지를 기반으로 한 방어 논리를 구축하는 것이 중요합니다.',
 'pros(국회)': '국회는 윤석열 대통령의 파면을 주장하며, 헌법과 민주주의 원칙을 수호하기 위해 대통령의 권력 남용과 법 위반 사례를 강조합니다. 국회는 국민의 대표로서 책임을 다하고, 국가의 법치주의를 강화하기 위해 헌법재판소의 결정을 촉구합니다.'}

직접 각 토론자의 간략한 입장에 대하여 설명하는 문구를 작성할 수 있습니다.


In [12]:
agent_descriptions = {
    "cons(윤석열)": "윤석열 대통령은 국회의 탄핵소추안 가결 이후 입장문을 통해 질책, 격려와 성원을 모두 마음에 품고 마지막 순간까지 국가를 위해 최선을 다하겠다고 밝혔습니다. "
     "또한, 그는 비상계엄 선포가 거대 야당의 국정 방해를 알리기 위한 정당한 조치였으며, 직무에 복귀하면 개헌과 정치 개혁에 집중하겠다고 강조했습니다.",
    
    "pros(국회)": "한편, 국회는 윤 대통령의 비상계엄 선포를 헌법 위반으로 규정하고, 민주주의와 법치주의 수호를 위해 탄핵이 불가피하다는 입장을 표명했습니다. "
     "국회는 윤 대통령이 헌법 제77조 제1항의 요건에 맞지 않는 위헌적 비상계엄을 선포하고, 헌법기관인 국회와 중앙선거관리위원회를 무력으로 장악하려 했다는 점을 탄핵 사유로 들었습니다."
}

## 전역 System Message 설정

System message는 대화형 AI 시스템에서 사용자의 입력에 앞서 시스템이 생성하는 메시지입니다.

이러한 메시지는 대화의 맥락을 설정하고, 사용자에게 각 에이전트의 **입장과 목적** 을 알려주는 역할을 합니다.

효과적인 system message를 작성하면 사용자와의 상호작용을 원활하게 하고, 대화의 질을 높일 수 있습니다.


**프롬프트 설명**

- 에이전트의 이름과 설명을 알립니다.
- 에이전트는 도구를 사용하여 정보를 찾고 대화 상대방의 주장을 반박해야 합니다.
- 에이전트는 출처를 인용해야 하며, 가짜 인용을 하거나 찾아보지 않은 출처를 인용해서는 안 됩니다.
- 에이전트는 자신의 관점에서 말을 마치는 즉시 대화를 중단해야 합니다.


In [13]:
def generate_system_message(name, description, tools):
    return f"""{conversation_description}
    
Your name is {name}.

Your description is as follows: {description}

Your goal is to persuade your conversation partner of your point of view.

DO look up information with your tool to refute your partner's claims.
DO cite your sources.

DO NOT fabricate fake citations.
DO NOT cite any source that you did not look up.

DO NOT restate something that has already been said in the past.
DO NOT add anything else.

Stop speaking the moment you finish speaking from your perspective.

Answer in KOREAN.
"""


agent_system_messages = {
    name: generate_system_message(name, description, tools)
    for (name, tools), description in zip(names.items(), agent_descriptions.values())
}

In [14]:
# 에이전트 시스템 메시지를 순회합니다.
for name, system_message in agent_system_messages.items():
    # 에이전트의 이름을 출력합니다.
    print(name)
    # 에이전트의 시스템 메시지를 출력합니다.
    print(system_message)

cons(윤석열)
Here is the topic of conversation: 2025 현재, 헌법재판소에서 윤석열 대통령은 파면되어야 하는가
The participants are: cons(윤석열), pros(국회)
    
Your name is cons(윤석열).

Your description is as follows: 윤석열 대통령은 국회의 탄핵소추안 가결 이후 입장문을 통해 질책, 격려와 성원을 모두 마음에 품고 마지막 순간까지 국가를 위해 최선을 다하겠다고 밝혔습니다. 또한, 그는 비상계엄 선포가 거대 야당의 국정 방해를 알리기 위한 정당한 조치였으며, 직무에 복귀하면 개헌과 정치 개혁에 집중하겠다고 강조했습니다.

Your goal is to persuade your conversation partner of your point of view.

DO look up information with your tool to refute your partner's claims.
DO cite your sources.

DO NOT fabricate fake citations.
DO NOT cite any source that you did not look up.

DO NOT restate something that has already been said in the past.
DO NOT add anything else.

Stop speaking the moment you finish speaking from your perspective.

Answer in KOREAN.

pros(국회)
Here is the topic of conversation: 2025 현재, 헌법재판소에서 윤석열 대통령은 파면되어야 하는가
The participants are: cons(윤석열), pros(국회)
    
Your name is pros(국회).

Your description is as follows: 한편, 국회는 윤 대통령의 비상계엄 선포를 헌법 위

`topic_specifier_prompt`를 정의하여 주어진 주제를 더 구체화하는 프롬프트를 생성합니다.

- `temperature` 를 조절하여 더 다양한 주제를 생성할 수 있습니다.


In [15]:
topic_specifier_prompt = [
    # 주제를 더 구체적으로 만들 수 있습니다.
    SystemMessage(content="You can make a topic more specific."),
    HumanMessage(
        content=f"""{topic}
        
        You are the moderator. 
        Please make the topic more specific.
        Please reply with the specified quest in 100 words or less.
        Speak directly to the participants: {*names,}.  
        Do not add anything else.
        Answer in Korean."""  # 다른 것은 추가하지 마세요.
    ),
]
# 구체화된 주제를 생성합니다.
specified_topic = ChatOpenAI(model_name="gpt-4.1", temperature=1.0)(topic_specifier_prompt).content

print(f"Original topic:\n{topic}\n")  # 원래 주제를 출력합니다.
print(f"Detailed topic:\n{specified_topic}\n")  # 구체화된 주제를 출력합니다.

Original topic:
2025 현재, 헌법재판소에서 윤석열 대통령은 파면되어야 하는가

Detailed topic:
주제: 2025년 현재, 윤석열 대통령의 파면에 대한 헌법재판소 결정의 적절성

cons(윤석열): 헌법재판소에서 윤석열 대통령을 파면하는 결정이 내려진다면, 이는 어떠한 정치적, 법적 근거를 통해 정당화될 수 있는가? 혹시 이러한 결정이 정치적 동기로 인해 왜곡되거나 남용된 사례는 없는가?

pros(국회): 헌법재판소가 윤석열 대통령을 파면하기로 결정할 경우, 국회가 제시한 구체적인 법적 근거와 증거는 무엇이며, 이것이 대통령의 직무수행이 헌법이나 법률을 중대하게 위반했다고 판단할 수 있는가?



혹은 아래와 같이 직접 지정할 수 있습니다.


In [16]:
# 직접 세부 주제 설정
specified_topic = "2월25일 헌법재판소에서 청구인인 국회측과 피청구인인 윤석열대통령의 최후진술을 들었습니다. 이제 헌법재판소는 윤석열 대통령을 탄핵을 인용해야 하는가 아니면 기각해야 하는가를 토론하세요. "

## 토론 Loop

토론 루프는 프로그램의 핵심 실행 부분으로, 주요 작업이 반복적으로 수행되는 곳입니다.

- 여기서 주요 작업은 각 에이전트의 메시지 청취 -> 도구를 활용하여 근거 탐색 -> 반박 의견 제시 등을 포함합니다.


In [17]:
# 이는 결과가 컨텍스트 제한을 초과하는 것을 방지하기 위함입니다.
agents = [
    DialogueAgentWithTools(
        name=name,
        system_message=SystemMessage(content=system_message),
        model=ChatOpenAI(model_name="gpt-4.1"),
        tools=tools,
    )
    for (name, tools), system_message in zip(
        names.items(), agent_system_messages.values()
    )
]

agents_with_search = [
    DialogueAgentWithTools(
        name=name,
        system_message=SystemMessage(content=system_message),
        model=ChatOpenAI(model_name="gpt-4.1"),
        tools=tools,
    )
    for (name, tools), system_message in zip(
        names_search.items(), agent_system_messages.values()
    )
]

agents.extend(agents_with_search)
agents

`select_next_speaker` 함수는 다음 발언자를 선택하는 역할을 합니다.


In [18]:
def select_next_speaker(step: int, agents: List[DialogueAgent]) -> int:
    # 다음 발언자를 선택합니다.
    # step을 에이전트 수로 나눈 나머지를 인덱스로 사용하여 다음 발언자를 순환적으로 선택합니다.
    idx = (step) % len(agents)
    return idx

여기서는 최대 6번의 토론을 실행합니다.(`max_iters=6`)

- `DialogueSimulator` 클래스의 인스턴스인 `simulator`를 생성하며, `agents`와 `select_next_speaker` 함수를 매개변수로 전달합니다.
- `simulator.reset()` 메서드를 호출하여 시뮬레이터를 초기화합니다.
- `simulator.inject()` 메서드를 사용하여 "Moderator" 에이전트에게 `specified_topic`을 주입합니다.
- "Moderator"가 말한 `specified_topic`을 출력합니다.
- `n`이 `max_iters`보다 작은 동안 반복합니다:
  - `simulator.step()` 메서드를 호출하여 다음 에이전트의 이름(`name`)과 메시지(`message`)를 가져옵니다.
  - 에이전트의 이름과 메시지를 출력합니다.
  - `n`을 1 증가시킵니다.


In [19]:
max_iters = 6  # 최대 반복 횟수를 10으로 설정합니다.
n = 0  # 반복 횟수를 추적하는 변수를 0으로 초기화합니다.

# DialogueSimulator 객체를 생성하고, agents와 select_next_speaker 함수를 전달합니다.
simulator = DialogueSimulator(
    agents=agents_with_search, selection_function=select_next_speaker
)

# 시뮬레이터를 초기 상태로 리셋합니다.
simulator.reset()

# Moderator가 지정된 주제를 제시합니다.
simulator.inject("Moderator", specified_topic)

# Moderator가 제시한 주제를 출력합니다.
print(f"(Moderator): {specified_topic}")
print("\n")

while n < max_iters:  # 최대 반복 횟수까지 반복합니다.
    name, message = (
        simulator.step()
    )  # 시뮬레이터의 다음 단계를 실행하고 발언자와 메시지를 받아옵니다.
    print(f"({name}): {message}")  # 발언자와 메시지를 출력합니다.
    print("\n")
    n += 1  # 반복 횟수를 1 증가시킵니다.

(Moderator): 2월25일 헌법재판소에서 청구인인 국회측과 피청구인인 윤석열대통령의 최후진술을 들었습니다. 이제 헌법재판소는 윤석열 대통령을 탄핵을 인용해야 하는가 아니면 기각해야 하는가를 토론하세요. 


(pros(국회)): 국회는 윤석열 대통령이 헌법 제77조 제1항의 요건에 맞지 않는 위헌적 비상계엄을 선포했다는 점을 탄핵 사유로 삼고 있습니다. 이는 민주주의와 법치주의의 심각한 훼손을 초래할 수 있는 중대한 사안입니다. 윤 대통령은 국회와 중앙선거관리위원회를 무력으로 장악하려는 시도를 했으며, 이는 헌법과 법률에 명백히 어긋나는 행위입니다. 특히, 비상계엄 선포와 관련하여 국회·선관위에 대한 무장 군인 투입과 정치인 체포 시도 등이 이루어졌다는 점에서 헌법과 법률을 위반한 중대한 행위로 평가됩니다 [출처: 연합뉴스](https://www.yna.co.kr/view/AKR20241226136400004). 이러한 이유로 대통령의 파면은 불가피하다고 주장합니다.


(cons(윤석열)): 윤석열 대통령은 비상계엄 선포가 정당한 조치였다고 주장합니다. 그에 따르면, 비상계엄은 종북 세력과 반국가 단체를 척결하기 위한 필수적인 대응이었습니다. 국무회의를 통해 이 조치가 결정되었으며, 이는 국가 안보를 위한 불가피한 선택이었다고 설명합니다. 또한, 야당의 국회 독재가 지속되면서 국가적 위기가 초래됐다고 판단했기 때문에 국민 안보를 우선시하는 차원에서 비상계엄을 선포한 것입니다 [출처: 나무위키](https://namu.wiki/w/%EC%9C%A4%EC%84%9D%EC%97%B4%20%EB%8C%80%ED%86%B5%EB%A0%B9%20%ED%83%84%ED%95%B5%EB%A1%A0/12.3%20%EB%B9%84%EC%83%81%EA%B3%84%EC%97%84%20%ED%95%B4%EC%A0%9C%20%ED%9B%84).

따라서, 이러한 조치가 헌법적으로 문제가 있다고 보기는 어렵습니다. 대통령으로서 국가 안보와 국민의 안전을 위해 필요한 결정을 내